# LLM Fine-Tuning Deep Dive, Part 3 of 3: Comparison & Decision

> **This is Part 3 of a three-notebook fine-tuning arc:**
>
> 1. [Part 1: Data-based techniques](01-llm-finetuning-data-techniques.ipynb) -- continued
>    pretraining (full FT), SFT (LoRA), preference alignment / DPO (LoRA).
> 2. [Part 2: Parameter-based techniques + QLoRA & quantization](02-llm-finetuning-parameter-techniques.ipynb)
>    -- full fine-tuning, partial freezing, LoRA, QLoRA, and a real post-training quantization demo.
> 3. **Part 3 (this notebook): Comparison & decision** -- head-to-head evaluation of all six trained
>    checkpoints, held-out perplexity, an ablation study, and the final call on what Riverside House
>    actually deploys.
>
> **Recap:** Parts 1 and 2 trained and saved six checkpoints to `./checkpoints/` on disk (continued
> pretraining / full FT, instruction-tuned LoRA, DPO-aligned LoRA, partial freezing, and LoRA
> continued pretraining), plus discussed QLoRA and quantization without training a seventh. This
> notebook reloads all six from disk -- fresh Python objects, not the same in-memory instances Parts
> 1-2 trained, since kernels don't share memory across notebooks -- and puts them head-to-head.

## Table of Contents (Part 3)

1. [Setup: Reloading All Six Trained Checkpoints](#setup-reloading-all-six-trained-checkpoints)
2. [Comparing All Six Techniques](#comparing-all-six-techniques)
3. [Decision Time: Which Model Does Riverside Actually Deploy?](#decision-time-which-model-does-riverside-actually-deploy)
4. [Automated Corpus Knowledge Tests: All Models](#automated-corpus-knowledge-tests-all-models)
5. [Deep Dive: Token Probability Analysis](#deep-dive-what-actually-changed-token-probability-analysis)
6. [Held-Out Perplexity](#held-out-perplexity-the-number-riverside-actually-needs)
7. [Technique Combination Grid: Data × Parameter](#technique-combination-grid-data--parameter)
8. [Ablation Study: What Happens If You Skip a Stage?](#ablation-study-what-happens-if-you-skip-a-stage)
9. [What This Notebook Covered (and What It Didn't)](#what-this-fine-tuning-arc-covered-and-what-it-didnt)
10. [Further Reading & Scaling Up](#further-reading--scaling-up)
11. [The Decision: What Do We Actually Hand to Riverside House?](#the-decision-what-do-we-actually-hand-to-riverside-house)

---

## Setup: Reloading All Six Trained Checkpoints

Everything below depends on six models that were trained across Parts 1 and 2, in different kernel
sessions. Rather than assume any of those kernels are still alive, this section reloads every
checkpoint fresh from `./checkpoints/`. Note: `./checkpoints/instruction-lora` was saved _before_
DPO ran in Part 1, `./checkpoints/preference-dpo` _after_ — so reloading both here gives the true
pre/post-DPO comparison.


## Choose the Training Path

![Fine-tuning decision matrix matching adaptation needs to continued pretraining, SFT plus LoRA, DPO plus LoRA, or full fine-tuning](images/finetuning-decision-matrix.png)

Use this as a first-pass decision aid, then validate the choice against held-out task metrics, safety constraints, available data, and deployment cost.


> **PyTorch → Keras:** `torch.cuda.is_available()` + `.to(device)` explicitly move a model/tensors to
> GPU or CPU, `AutoModelForCausalLM.from_pretrained(...)` loads pretrained weights, and
> `model.generate(...)` run inside `torch.no_grad()` performs autoregressive decoding without tracking
> gradients (nothing to backprop through during inference). **Keras/TF equivalent:** TensorFlow places
> ops on GPU automatically (explicit placement is `tf.device(...)`, rarely needed); the loading call
> would be `TFAutoModelForCausalLM.from_pretrained(...)` followed by the same `.generate(...)` method --
> Keras/TF has no separate "no_grad" context since inference doesn't build a gradient tape by default.


In [ ]:
# Re-establishing Parts 1-2's foundations -- see Part 1 for the reasoning behind each choice below.
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_NAME = "gpt2-medium"

# Prefer a GPU if one is visible to PyTorch, otherwise fall back to CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load the BPE tokenizer paired with MODEL_NAME's pretrained vocabulary
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:

    # GPT-2 never defined a pad token, so borrow EOS for padding
    tokenizer.pad_token = tokenizer.eos_token

# Load the pretrained weights and move them onto the selected device
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
PROMPT = "Aria Voss stared at the signal counting itself out in prime numbers and"
INSTRUCTION_PREFIX = "Continue the fiction narrative in the same style:\n\n"


# Generate a continuation, stripping the echoed prompt from the result
def generate(model, prompt, max_new_tokens=60):
    """Generate a continuation, returning only the new tokens (full walkthrough is in Part 1)."""
    # Switch to inference mode (disables dropout, etc.)
    model.eval()

    # Tokenize the prompt and move the tensors onto the model's device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # Remember where the prompt ends so it can be sliced off the generated output
    prompt_len = inputs["input_ids"].shape[1]

    # No gradients needed for inference -- saves memory and compute
    with torch.no_grad():

        # Autoregressively sample new tokens with nucleus (top-p) sampling
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Drop the echoed prompt tokens and decode only the newly generated ones
    completion = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()

    # Guard against an empty string if the model sampled EOS as its very first token
    return (
        completion
        if completion
        else "[model stopped immediately — sampled EOS as first token]"
    )


print(f"Baseline completion (sanity check): {generate(base_model, PROMPT)}")

### Reloading the Five Fine-Tuned Checkpoints

Each PEFT-wrapped adapter (instruction-tuned LoRA, DPO, LoRA continued pretraining) gets its own fresh
base-model instance rather than sharing `base_model` above -- the same "every PEFT wrapper gets its
own base" rule Parts 1-2 followed throughout. `freeze_model`'s `requires_grad` flags are re-applied
after loading (see the comment below) since that bookkeeping isn't part of a saved checkpoint -- only
the trained weights are.


> **PyTorch → Keras:** `PeftModel.from_pretrained(base_model, path)` wraps a fresh base model with a
> saved LoRA adapter's weights; `named_parameters()` iterates `(name, tensor)` pairs so `requires_grad`
> can be toggled per-parameter (used here to re-apply the freeze pattern, since that bookkeeping isn't
> part of a saved checkpoint), and `p.numel()` counts a tensor's elements to total trainable params.
> **Keras/TF equivalent:** LoRA loading has no single standard TF API (usually a custom `tf.keras.Model`
> subclass or a TF-specific PEFT integration); freezing is coarser-grained -- `layer.trainable = False`
> per layer rather than per-parameter -- and element counts come from `tf.size(variable)`.


In [ ]:
# Reload every checkpoint Parts 1-2 saved to disk.
# Load the fully fine-tuned continued-pretraining checkpoint
non_instruct_ckpt = AutoModelForCausalLM.from_pretrained(
    "./checkpoints/non-instruction-full"
).to(device)

# Fresh base model + the instruction-tuned LoRA adapter wrapped around it
instruct_base_reload = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
instruct_lora_model = PeftModel.from_pretrained(
    instruct_base_reload, "./checkpoints/instruction-lora"
).to(device)

# Fresh base model + the DPO-aligned policy adapter wrapped around it
dpo_base_reload = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
policy_model = PeftModel.from_pretrained(
    dpo_base_reload, "./checkpoints/preference-dpo"
).to(device)

# Load the partially-frozen checkpoint
freeze_model = AutoModelForCausalLM.from_pretrained("./checkpoints/partial-freeze").to(
    device
)

# requires_grad isn't part of a saved checkpoint -- reapply Part 2's freeze pattern purely so the
# trainable-parameter percentage below is accurate (the weights themselves are already correct).
n_layers = freeze_model.config.n_layer

# Unfreeze only the last quarter of blocks (at least 2), matching Part 2's rule
unfreeze_from = n_layers - max(2, n_layers // 4)

# Start every parameter frozen
for name, param in freeze_model.named_parameters():
    param.requires_grad = False

# Re-unfreeze the trailing blocks plus the final norm/head, mirroring Part 2's config
for name, param in freeze_model.named_parameters():
    if any(f"h.{i}." in name for i in range(unfreeze_from, n_layers)):
        param.requires_grad = True
    if "ln_f" in name or "lm_head" in name:
        param.requires_grad = True

# Fresh base model + the LoRA continued-pretraining adapter wrapped around it
lora_pt_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
lora_pt_model = PeftModel.from_pretrained(lora_pt_base, "./checkpoints/peft-lora").to(
    device
)

# Switch every reloaded model to inference mode
for m in (
    non_instruct_ckpt,
    instruct_lora_model,
    policy_model,
    freeze_model,
    lora_pt_model,
):
    m.eval()

# Recompute the trainable-parameter percentages Part 2 measured, from these reloaded models --
# needed later for the Technique Combination Grid.
# Full fine-tuning updates every parameter in the model
total_params = sum(p.numel() for p in base_model.parameters())
full_ft_params = total_params

# Count only the parameters partial freezing left trainable
partial_ft_params = sum(p.numel() for p in freeze_model.parameters() if p.requires_grad)

# Count only the LoRA adapter's trainable parameters
lora_params = sum(
    p.numel() for p in instruct_lora_model.parameters() if p.requires_grad
)
param_counts = [full_ft_params, partial_ft_params, lora_params]

# Convert each raw trainable-parameter count into a percentage of the total model size
param_pcts = [count / total_params * 100 for count in param_counts]

print("Reloaded all six checkpoints (baseline + 5 fine-tuned):")
print(f"  Full fine-tuning:     {full_ft_params:,} trainable ({param_pcts[0]:.2f}%)")
print(f"  Partial freezing:     {partial_ft_params:,} trainable ({param_pcts[1]:.2f}%)")
print(f"  LoRA:                 {lora_params:,} trainable ({param_pcts[2]:.2f}%)")

In [ ]:
# Corpus loader (needed for the held-out perplexity + ablation sections further down) and
# visualization imports used throughout this notebook.
# Resolve the notebook's own directory, falling back to cwd if the VS Code variable is unavailable
try:
    _notebook_dir = Path(__vsc_ipynb_file__).parent  # type: ignore[name-defined]
except NameError:
    try:
        _notebook_dir = Path(__file__).parent
    except NameError:
        _notebook_dir = Path.cwd()

CONTENT_DIR = _notebook_dir / "content"

# If the sibling content folder isn't found here, try resolving it from the repo root instead
if not CONTENT_DIR.exists():
    _fallback = Path.cwd() / "learning" / "genai" / "04-llm" / "content"
    if _fallback.exists():
        CONTENT_DIR = _fallback

print(f"Content directory: {CONTENT_DIR.absolute()}")

# Novel corpus filenames keyed by genre alias, shared by every corpus-loading helper below
NOVELS = {
    "scifi": "the-weight-of-distant-light",
    "fantasy": "the-tidebound-accord",
    "mystery": "the-cartographers-cipher",
    "historical": "the-silk-merchants-daughter",
    "cyberpunk": "neural-drift",
    "horror": "the-hollow-beneath",
    "literary": "the-weight-of-tides",
}


# Load paragraphs long enough to be useful, from each requested novel's earliest chapters
def load_corpus_paragraphs(novels=None, max_chapters=10, min_len=200):
    """Load paragraphs from the multi-novel corpus (identical to Parts 1-2)."""
    if novels is None:
        novels = list(NOVELS.keys())
    paragraphs = []

    # Walk every requested novel and pull paragraphs from its earliest chapters
    for alias in novels:
        novel_dir = NOVELS.get(alias)
        if not novel_dir:
            continue
        novel_path = CONTENT_DIR / novel_dir
        if not novel_path.exists():
            continue
        chapter_files = sorted(novel_path.glob("chapter-*.txt"))[:max_chapters]
        for path in chapter_files:
            text = path.read_text(encoding="utf-8")

            # Split on blank lines and keep only paragraphs above the minimum length
            for para in text.split("\n\n"):
                para = para.strip().replace("\n", " ")
                if len(para) >= min_len:
                    paragraphs.append(para)
    return paragraphs


import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings

# Silence noisy library warnings so they don't clutter notebook output
warnings.filterwarnings("ignore")

# Keep figure resolution/font size consistent across every plot in this notebook
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})
sns.set_theme(style="whitegrid", palette="muted")

print("Corpus loader and visualization imports ready.")


---

## Comparing All Six Techniques

> **How to read this comparison:** The first three rows (Concepts 1–3) were all trained with
> _different data objectives_ but the same-or-similar parameter budgets (full FT for pretraining,
> LoRA for the others). The last four rows (Concepts 4–7) used _different parameter strategies_ all
> on the same continued-pretraining objective. Comparing within each group is apples-to-apples.
> Comparing across groups — e.g. the SFT model vs. the partial-freeze model — changes two variables
> at once. See the [Technique Combination Grid](#technique-combination-grid-data--parameter) below
> for the full 3 × 3 matrix that disentangles the axes.

### Same data objective, different parameter strategy (Concepts 4–6 vs. Concept 1)

Rows 4–6 all used continued pretraining (same data objective as Concept 1) but vary how many
weights updated. This group is the controlled parameter-axis comparison.

### Same parameter strategy, different data objective (Concepts 1–3)

Rows 1–3 all used LoRA (or full FT for pretraining) but vary what the model was trained _to do_.
This group is the controlled data-axis comparison.

### Cross-axis: mixing data and parameter choices

The remaining combinations (e.g. SFT + full FT, DPO + partial freeze) weren't trained in this run.
The heatmap below fills in the five cells we did train; the four grey cells are inference about what
those combinations would likely produce.

| Technique                                   | Data objective        | Parameter strategy   | Trainable Params | Best For                                    |
| ------------------------------------------- | --------------------- | -------------------- | ---------------- | ------------------------------------------- |
| Concept 1: Continued pretraining            | Next-token prediction | Full FT (100%)       | 100%             | Absorbing vocabulary/style/facts            |
| Concept 2: Instruction tuning (SFT)         | Instruction-following | LoRA                 | well under 1%    | Teaching task-following behavior            |
| Concept 3: Preference alignment (DPO)       | Human preference      | LoRA                 | well under 1%    | Aligning to human preference                |
| Concept 4: Full fine-tuning                 | Next-token prediction | Full FT (100%)       | 100%             | Max quality, abundant compute               |
| Concept 5: Partial freezing                 | Next-token prediction | ~21% of params       | ~21%             | Middle ground on compute/quality            |
| Concept 6: LoRA (continued pretraining)     | Next-token prediction | LoRA                 | well under 1%    | Cheapest, swappable, lowest forgetting risk |
| Concept 7: QLoRA (Part 2, not trained here) | Any                   | QLoRA (4-bit + LoRA) | well under 1%    | Largest models, GPU required                |

In production, a realistic pipeline stacks the data-based stages in order (continued pretraining →
instruction tuning → preference alignment) while choosing the parameter strategy based on compute
budget — most commonly LoRA or QLoRA throughout for models larger than a few billion parameters.

## Decision Time: Which Model Does Riverside Actually Deploy?

We now have six trained checkpoints and a laptop CPU. Before Riverside's editors and employees get
access to any of them, we need more evidence than "it read fine to me" -- so this section does two
things: a **qualitative** side-by-side read on real prompts from the catalog, and then a
**quantitative** held-out perplexity check (further down) that scores every checkpoint on manuscript
paragraphs _none of them trained on_ -- the closest thing we have to "how will it behave on a chapter
it hasn't memorized."

### Side-by-Side: Every Checkpoint on the Same Prompt

Let's compare the baseline against every fine-tuned variant trained above, on the same prompt from the
catalog.


> **PyTorch → Keras:** `AutoModelForCausalLM.from_pretrained("./checkpoints/...")` reloads a saved
> fine-tuned checkpoint from disk into a fresh `torch.nn.Module`, then `.to(device)` places it on
> GPU/CPU before the loop below calls the `generate()` helper defined earlier on each model in turn.
> **Keras/TF equivalent:** `TFAutoModelForCausalLM.from_pretrained(path)` loads the same checkpoint
> format into a `tf.keras.Model`; TensorFlow doesn't need an explicit `.to(device)` call since device
> placement is handled by default device scoping (or `tf.distribute` for multi-device setups) instead.


In [ ]:
# Select a smaller subset of prompts for comparison across all models
COMPARISON_PROMPTS = {
    "scifi": "Aria Voss checked the Meridian's Promise status panel and",
    "fantasy": "Kerra Valmont felt all five tides simultaneously as",
    "mystery": "Elena Voss studied the 1879 survey map and realized",
    "cyberpunk": "In the Lower Stacks of Neo-Shanghai, Kai Chen",
}

# Load the continued pretraining checkpoint for comparison
non_instruct_ckpt = AutoModelForCausalLM.from_pretrained(
    "./checkpoints/non-instruction-full"
).to(device)

# All six checkpoints under comparison, keyed by display name
models_to_test = {
    "Baseline (no fine-tuning)": base_model,
    "Continued pretraining (full FT)": non_instruct_ckpt,
    "Instruction-tuned (LoRA)": instruct_lora_model,
    "Preference-aligned (DPO)": policy_model,
    "Partial fine-tuning": freeze_model,
    "PEFT LoRA continued pretraining": lora_pt_model,
}

print("=" * 80)
print("CORPUS KNOWLEDGE COMPARISON ACROSS ALL FINE-TUNING TECHNIQUES")
print("=" * 80)

# Run every model against every genre prompt to compare corpus knowledge side by side
for prompt_name, prompt in COMPARISON_PROMPTS.items():
    print(f"\n{'' * 80}")
    print(f'Prompt ({prompt_name}): "{prompt}"')
    print(f"{'' * 80}")

    for model_name, model in models_to_test.items():

        # Instruction/preference models expect the training-time instruction prefix
        uses_prefix = "Instruction" in model_name or "Preference" in model_name
        effective_prompt = (
            (INSTRUCTION_PREFIX + prompt + "\n\n") if uses_prefix else prompt
        )
        output = generate(model, effective_prompt, max_new_tokens=60)

        # Truncate long completions so the comparison stays readable
        output_display = (output[:120] + "...") if len(output) > 120 else output

        prefix_note = " (+instruction prefix)" if uses_prefix else ""
        print(f"\n[{model_name}]")
        print(
            f"  Input{prefix_note}: {repr(prompt[:80])}{'...' if len(prompt) > 80 else ''}"
        )
        print(f"  Output: {output_display}")

print("\n" + "=" * 80)
print("ANALYSIS:")
print("- Baseline should produce generic, off-corpus continuations")
print("- Fine-tuned models should recognize characters/settings and continue in-world")
print("- Compare vocabulary, narrative coherence, and genre-appropriate style")
print("=" * 80)

## Automated Corpus Knowledge Tests: All Models

Now let's run the corpus-specific test prompts on every fine-tuned checkpoint and compare how well
each technique absorbed the domain knowledge. We'll use one representative prompt from each novel and
compare baseline vs. all fine-tuned variants.


In [ ]:
# Build the shared prompt with the instruction prefix, used by the instruction/preference models
instruct_prompt = INSTRUCTION_PREFIX + PROMPT + "\n\n"
print(f"Shared prompt (plain models)       : {repr(PROMPT)}")
print(
    f"Shared prompt (instruction models) : (+ instruction prefix) {repr(PROMPT[:60])}..."
)
print()

print("=== Baseline (no fine-tuning) ===")
print(f"  Input : {repr(PROMPT)}")
print(f"  Output: {generate(base_model, PROMPT)}")
print()

print("=== Non-instructional continued pretraining (full fine-tune) ===")
print(f"  Input : {repr(PROMPT)}")
print(f"  Output: {generate(non_instruct_ckpt, PROMPT)}")
print()

print("=== Instruction-tuned (LoRA) ===")
print(f"  Input : (+instruction prefix) {repr(PROMPT[:60])}...")
print(f"  Output: {generate(instruct_lora_model, instruct_prompt)}")
print()

print("=== Preference-aligned (DPO on top of the instruction-tuned LoRA adapter) ===")
print(f"  Input : (+instruction prefix) {repr(PROMPT[:60])}...")
print(f"  Output: {generate(policy_model, instruct_prompt)}")
print()

print("=== Partial fine-tuning (layer freezing) ===")
print(f"  Input : {repr(PROMPT)}")
print(f"  Output: {generate(freeze_model, PROMPT)}")
print()

print("=== Parameter-efficient (LoRA continued pretraining) ===")
print(f"  Input : {repr(PROMPT)}")
print(f"  Output: {generate(lora_pt_model, PROMPT)}")
print()

## Deep Dive: What Actually Changed? Token Probability Analysis

Let's go deeper than just comparing generated text. We'll look at **exactly how the model's internal
probability distribution shifted** after fine-tuning. This is the transformers notebook style:
concrete, numerical, and visual.

**Question:** After fine-tuning on our 7-novel corpus, how much more likely is the model to predict
domain-specific words vs. generic words?

We'll compare the baseline model vs. the fine-tuned model on a **single next-token prediction** for a
domain-specific prompt.


> **PyTorch → Keras:** `model.eval()` switches dropout/batch-norm-style layers to inference behavior;
> `torch.no_grad()` disables gradient tracking for the forward pass below; calling `model(**inputs)`
> runs a forward pass and returns `outputs.logits` (raw scores), and `F.softmax(logits, dim=-1)`
> (from `torch.nn.functional`) converts those logits into a probability distribution over the vocabulary.
> **Keras/TF equivalent:** Keras layers infer train/inference behavior automatically (or via a
> `training=False` argument) instead of an explicit `.eval()` call, and there's no separate "no_grad"
> context since plain forward calls outside a `GradientTape` don't track gradients; the softmax step is
> `tf.nn.softmax(logits, axis=-1)` -- same idea, `axis` instead of `dim`.


In [ ]:
# Token probability analysis: before vs after fine-tuning
import torch.nn.functional as F

# Prompt: "Aria Voss checked the Meridian's Promise and"
prompt_for_analysis = "Aria Voss checked the Meridian's Promise and"

# Words we expect to be more likely after fine-tuning (domain-specific)
domain_words = [
    "saw",
    "discovered",
    "noted",
    "realized",
    "found",
    "detected",
    "confirmed",
]

# Generic words that might be likely in base model
generic_words = ["the", "a", "then", "he", "she", "was", "said"]


# Score a fixed list of candidate next-token words under one model
def get_next_token_probs(model, prompt, candidate_words):
    """Get the probability of specific next tokens given a prompt."""
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :]  # logits for the next token
        probs = F.softmax(logits, dim=-1)

    results = {}

    # Score every candidate word's probability under this model
    for word in candidate_words:

        # Tokenize the word (might be multi-token, take first)
        word_ids = tokenizer.encode(" " + word, add_special_tokens=False)
        if len(word_ids) > 0:
            token_id = word_ids[0]
            results[word] = probs[token_id].item()

    return results


# Get probabilities from baseline and fine-tuned models
print("Computing token probabilities...")
baseline_domain_probs = get_next_token_probs(
    base_model, prompt_for_analysis, domain_words
)
baseline_generic_probs = get_next_token_probs(
    base_model, prompt_for_analysis, generic_words
)

finetuned_domain_probs = get_next_token_probs(
    non_instruct_ckpt, prompt_for_analysis, domain_words
)
finetuned_generic_probs = get_next_token_probs(
    non_instruct_ckpt, prompt_for_analysis, generic_words
)

# Set up a two-panel figure: domain words on the left, generic words on the right
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Domain-specific words
# Sort domain words by their fine-tuned probability, highest first
domain_words_sorted = sorted(
    baseline_domain_probs.keys(), key=lambda w: finetuned_domain_probs[w], reverse=True
)
x = np.arange(len(domain_words_sorted))
width = 0.35

# Convert each domain word's probability to a percentage for the bar chart
baseline_vals = [baseline_domain_probs[w] * 100 for w in domain_words_sorted]
finetuned_vals = [finetuned_domain_probs[w] * 100 for w in domain_words_sorted]

# Grouped bar chart comparing baseline vs fine-tuned probability per domain word
bars1 = ax1.bar(
    x - width / 2, baseline_vals, width, label="Baseline", alpha=0.8, color="steelblue"
)
bars2 = ax1.bar(
    x + width / 2, finetuned_vals, width, label="Fine-tuned", alpha=0.8, color="coral"
)

ax1.set_xlabel("Domain-Specific Next Token")
ax1.set_ylabel("Probability (%)")
ax1.set_title(
    "Domain Word Probabilities: Fine-Tuning Boosts Relevant Vocabulary",
    fontweight="bold",
)
ax1.set_xticks(x)
ax1.set_xticklabels(domain_words_sorted, rotation=45, ha="right")
ax1.legend()
ax1.grid(alpha=0.3, axis="y")

# Annotate each bar pair with its probability delta, skipping negligible shifts
for i, word in enumerate(domain_words_sorted):
    change = finetuned_vals[i] - baseline_vals[i]
    if abs(change) > 0.01:  # only annotate significant changes
        ax1.annotate(
            f"+{change:.2f}%" if change > 0 else f"{change:.2f}%",
            xy=(i + width / 2, finetuned_vals[i]),
            xytext=(0, 5),
            textcoords="offset points",
            fontsize=8,
            color="green" if change > 0 else "red",
            fontweight="bold",
        )

# Plot 2: Generic words
# Sort generic words by their baseline probability, highest first
generic_words_sorted = sorted(
    baseline_generic_probs.keys(), key=lambda w: baseline_generic_probs[w], reverse=True
)
x2 = np.arange(len(generic_words_sorted))

# Convert each generic word's probability to a percentage for the bar chart
baseline_gen = [baseline_generic_probs[w] * 100 for w in generic_words_sorted]
finetuned_gen = [finetuned_generic_probs[w] * 100 for w in generic_words_sorted]

# Grouped bar chart comparing baseline vs fine-tuned probability per generic word
bars3 = ax2.bar(
    x2 - width / 2, baseline_gen, width, label="Baseline", alpha=0.8, color="steelblue"
)
bars4 = ax2.bar(
    x2 + width / 2,
    finetuned_gen,
    width,
    label="Fine-tuned",
    alpha=0.8,
    color="lightgreen",
)

ax2.set_xlabel("Generic Next Token")
ax2.set_ylabel("Probability (%)")
ax2.set_title(
    "Generic Word Probabilities: Hypothesized to Stay Relatively Stable",
    fontweight="bold",
)
ax2.set_xticks(x2)
ax2.set_xticklabels(generic_words_sorted, rotation=45, ha="right")
ax2.legend()
ax2.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

# Print summary
print(f"\n{'=' * 80}")
print("Token Probability Analysis Summary:")
print(f"{'=' * 80}")
print(f"Prompt: '{prompt_for_analysis}'")
print()

# Domain words: find biggest increases
# Compute each domain word's probability delta from baseline to fine-tuned
domain_increases = {
    w: (finetuned_domain_probs[w] - baseline_domain_probs[w]) * 100
    for w in domain_words
}

# Keep only the three largest probability increases
top_increases = sorted(domain_increases.items(), key=lambda x: x[1], reverse=True)[:3]

print("Top domain word probability increases:")
for word, increase in top_increases:
    base_pct = baseline_domain_probs[word] * 100
    ft_pct = finetuned_domain_probs[word] * 100
    print(f"  '{word}': {base_pct:.3f}% → {ft_pct:.3f}% (+{increase:.3f}%)")

print()
print("Generic word stability:")
for word in generic_words_sorted[:3]:
    base_pct = baseline_generic_probs[word] * 100
    ft_pct = finetuned_generic_probs[word] * 100
    change = ft_pct - base_pct
    print(
        f"  '{word}': {base_pct:.3f}% → {ft_pct:.3f}% ({'↑' if change > 0 else '↓'}{abs(change):.3f}%)"
    )

print(f"{'=' * 80}")
print("Interpretation (computed from the run above, not assumed):")

# Count how many domain words actually gained probability after fine-tuning
domain_gained = sum(1 for w in domain_words if domain_increases[w] > 0)

# Average absolute probability shift for generic vs domain words, to compare stability
generic_avg_abs_shift = sum(
    abs(finetuned_generic_probs[w] - baseline_generic_probs[w]) * 100
    for w in generic_words
) / len(generic_words)
domain_avg_abs_shift = sum(abs(v) for v in domain_increases.values()) / len(
    domain_words
)

print(
    f"  • {domain_gained}/{len(domain_words)} domain words gained probability "
    f"(avg |shift| = {domain_avg_abs_shift:.2f}%)."
)
print(f"  • Generic words moved by avg |shift| = {generic_avg_abs_shift:.2f}%.")

# Branch the printed interpretation on the actual measured shift, not an assumed outcome
if generic_avg_abs_shift >= domain_avg_abs_shift * 0.5:
    print(
        "  • The 'generic words should stay stable' hypothesis did NOT clearly hold here: "
        "generic-word probability moved almost as much as domain-word probability. With only "
        "60 full-fine-tuning steps on a narrow corpus, the model is shifting its overall "
        "*register* (which common words it reaches for), not just adding a few new domain "
        "facts on top of an unchanged base distribution -- a good reminder that full "
        "fine-tuning can nudge general behavior even when that's not the intent."
    )
else:
    print(
        "  • Generic words stayed comparatively stable relative to the domain-word shift, "
        "consistent with the 'model absorbed vocabulary without forgetting general language' "
        "story."
    )
print(f"{'=' * 80}")


## Held-Out Perplexity: The Number Riverside Actually Needs

Every comparison so far has been qualitative -- read a paragraph, judge it by eye. That's fine for
building intuition, but it's not what convinces an IT lead to deploy a model company-wide. So: let's
score **all six checkpoints** on the same held-out paragraphs, taken from chapters deep in each novel
that **none of the training runs above ever saw** (every training cell used only the first few
chapters per novel; this held-out set starts at chapter 11). Lower loss / perplexity means the model
assigns higher probability to Riverside's actual prose -- the closest thing we have to "how will this
behave on a chapter it hasn't memorized."

One honest caveat before the numbers: the instruction-tuned and DPO models were trained to expect the
`INSTRUCTION_PREFIX` format, not to plainly continue raw prose (that's exactly the "Test 2: without
the prefix" pitfall from the instruction-tuning section). Scoring them here without that prefix is
deliberate -- it answers "how good is this model as a general house-style language model," which is
what the knowledge-base use case actually needs, and it's fair to expect instruction/DPO models to
look worse on this specific metric even if they're better at the tasks they were tuned for.


> **PyTorch → Keras:** passing `labels=enc["input_ids"]` into the model's forward call makes the
> Hugging Face model compute cross-entropy loss internally and return it as `out.loss` (a scalar
> tensor); `torch.no_grad()` skips gradient tracking since this is evaluation-only, and `.item()`
> pulls the plain Python float out of that 0-d tensor, which `math.exp(loss)` then turns into
> perplexity. **Keras/TF equivalent:** the TF counterpart model supports the same `labels=` convenience
> (`model(enc, labels=...)`), while plain Keras code would instead call
> `tf.keras.losses.SparseCategoricalCrossentropy()(y_true, logits)` and use `.numpy()` in place of
> `.item()` to extract the scalar.


In [ ]:
# Build a held-out set from LATER chapters -- every training run above used max_chapters<=4,
# so starting at chapter index 10 guarantees none of these paragraphs were trained on.
import math


# Load paragraphs from chapters no training run above ever saw
def load_holdout_paragraphs(start_chapter_index=10, chapters_per_novel=2, min_len=200):
    paragraphs = []

    # Pull a fixed window of later chapters from every novel
    for alias, novel_dir in NOVELS.items():
        novel_path = CONTENT_DIR / novel_dir
        chapter_files = sorted(novel_path.glob("chapter-*.txt"))
        holdout_files = chapter_files[
            start_chapter_index : start_chapter_index + chapters_per_novel
        ]
        for path in holdout_files:
            text = path.read_text(encoding="utf-8")

            # Split on blank lines and keep only paragraphs above the minimum length
            for para in text.split("\n\n"):
                para = para.strip().replace("\n", " ")
                if len(para) >= min_len:
                    paragraphs.append(para)
    return paragraphs


holdout_paragraphs = load_holdout_paragraphs()
print(
    f"Held-out set: {len(holdout_paragraphs)} paragraphs from chapter 11+ of each novel "
    f"-- unseen by every training run above."
)


# Average next-token cross-entropy loss over every held-out paragraph
def compute_holdout_loss(model, paragraphs, max_length=128):
    model.eval()
    losses = []

    # No gradients needed -- this is pure evaluation, not training
    with torch.no_grad():
        for para in paragraphs:
            enc = tokenizer(
                para, truncation=True, max_length=max_length, return_tensors="pt"
            ).to(device)

            # Passing labels=input_ids makes HF compute the causal-LM loss internally
            out = model(**enc, labels=enc["input_ids"])
            losses.append(out.loss.item())
    return sum(losses) / len(losses)


# All six checkpoints under held-out evaluation, keyed by display name
models_for_eval = {
    "Baseline (no fine-tuning)": base_model,
    "Full fine-tuning": non_instruct_ckpt,
    "Instruction-tuned (LoRA)": instruct_lora_model,
    "Preference-aligned (DPO)": policy_model,
    "Partial freezing": freeze_model,
    "LoRA continued pretraining": lora_pt_model,
}

print(f"\n{'=' * 70}\nHeld-out evaluation (lower is better):\n{'=' * 70}")
holdout_results = {}

# Score every checkpoint on the same held-out paragraphs and convert loss to perplexity
for name, model in models_for_eval.items():
    avg_loss = compute_holdout_loss(model, holdout_paragraphs)
    perplexity = math.exp(avg_loss)
    holdout_results[name] = {"loss": avg_loss, "perplexity": perplexity}
    print(f"  {name:<32} loss={avg_loss:6.3f}   perplexity={perplexity:8.1f}")
print(f"{'=' * 70}")

# Rank and visualize
# Sort checkpoints from best (lowest perplexity) to worst
ranked = sorted(holdout_results.items(), key=lambda kv: kv[1]["perplexity"])
fig, ax = plt.subplots(figsize=(10, 5))
names = [name for name, _ in ranked]
ppls = [res["perplexity"] for _, res in ranked]

# Highlight the best-performing checkpoint in a distinct color
bar_colors = ["mediumseagreen" if i == 0 else "steelblue" for i in range(len(ranked))]

# Horizontal bar chart ranking every checkpoint by held-out perplexity
ax.barh(names[::-1], ppls[::-1], color=bar_colors[::-1])
ax.set_xlabel("Held-out perplexity (lower = better fit to Riverside's prose)")
ax.set_title(
    "Held-Out Perplexity Across All Six Checkpoints\n(unseen chapters -- the closest thing to a deployment test)",
    fontsize=11,
    fontweight="bold",
)
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

print(
    f"\nBest held-out fit: '{ranked[0][0]}' (perplexity={ranked[0][1]['perplexity']:.1f})"
)
print(
    "Remember the caveat above: this metric rewards raw next-token prediction on plain prose, "
    "which structurally favors the continued-pretraining/full-fine-tuning family over the "
    "instruction/DPO models (which optimized for a different job). Use this number alongside the "
    "qualitative reads above, not instead of them -- it answers 'best house-style language model,' "
    "not 'best assistant for an editor to talk to.'"
)


## Technique Combination Grid: Data × Parameter

The two fine-tuning axes are **independent choices** — any data objective can be paired with any
parameter strategy. This notebook trained six checkpoints that cover five of the nine cells; the
grid below maps all nine combinations and fills in what the actual numbers say where we have them,
marking the rest as "not trained in this run."

|                                | **Full FT (100%)**            | **Partial Freeze (~21%)** | **LoRA (<1%)**                  |
| ------------------------------ | ----------------------------- | ------------------------- | ------------------------------- |
| **Continued pretraining**      | trained — `non_instruct_ckpt` | trained — `freeze_model`  | trained — `lora_pt_model`       |
| **Instruction tuning (SFT)**   | not trained                   | not trained               | trained — `instruct_lora_model` |
| **Preference alignment (DPO)** | not trained                   | not trained               | trained — `policy_model`        |

The cell below visualises these five cells as a heatmap of held-out perplexity and trainable-parameter
percentage, using the real numbers from this notebook's runs — the untrained cells are shown as `NaN`
and greyed out.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap

#  Axis labels
data_objectives = [
    "Continued\nPretraining",
    "Instruction\nTuning (SFT)",
    "Preference\nAlignment (DPO)",
]
param_strategies = ["Full FT\n(100%)", "Partial Freeze\n(~21%)", "LoRA\n(<1%)"]

#  Map each (data, param) cell to the checkpoint name in holdout_results
# Cells that were NOT trained in this notebook run are marked None.
checkpoint_map = {
    (0, 0): "Full fine-tuning",
    (0, 1): "Partial freezing",
    (0, 2): "LoRA continued pretraining",
    (1, 2): "Instruction-tuned (LoRA)",
    (2, 2): "Preference-aligned (DPO)",
}

#  Pull actual held-out perplexity from holdout_results
ppl_grid = np.full((3, 3), np.nan)
trained_mask = np.zeros((3, 3), dtype=bool)

# Fill in perplexity only for the cells that were actually trained; leave the rest as NaN
for (r, c), ckpt_name in checkpoint_map.items():
    if ckpt_name in holdout_results:
        ppl_grid[r, c] = holdout_results[ckpt_name]["perplexity"]
        trained_mask[r, c] = True

#  Trainable-parameter percentages (real values from earlier cells)
# Rows = data objective (all share the same param axis), Cols = param strategy
param_pct_row = [param_pcts[0], param_pcts[1], param_pcts[2]]  # [Full, Partial, LoRA]
param_grid = np.array([param_pct_row] * 3)  # same for every data row

#  Figure: two side-by-side heatmaps
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(
    "Data × Parameter Technique Grid — Five Trained Combinations\n"
    "(grey cells were not trained in this notebook run)",
    fontsize=13,
    fontweight="bold",
)


def draw_grid(ax, values, fmt, title, cmap_name, label):
    """Render one heatmap cell; grey out untrained cells."""
    # Build a masked array for matplotlib so NaN cells render visibly distinct
    display_vals = np.where(trained_mask, values, np.nan)

    # Custom colormap with grey for NaN
    cmap = plt.get_cmap(cmap_name).copy()
    cmap.set_bad(color="#d3d3d3")

    # Only the trained cells contribute to the color scale range
    valid = display_vals[trained_mask]
    vmin, vmax = (valid.min(), valid.max()) if valid.size > 1 else (0, 1)

    # Render the grid as a heatmap using the masked colormap
    im = ax.imshow(display_vals, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")

    # Annotate each cell
    for r in range(3):
        for c in range(3):
            if trained_mask[r, c]:
                txt = fmt.format(display_vals[r, c])
                color = (
                    "white"
                    if display_vals[r, c] < (vmin + (vmax - vmin) * 0.6)
                    else "black"
                )
                ax.text(
                    c,
                    r,
                    txt,
                    ha="center",
                    va="center",
                    fontsize=11,
                    fontweight="bold",
                    color=color,
                )
            else:
                ax.text(
                    c,
                    r,
                    "—\n(not trained)",
                    ha="center",
                    va="center",
                    fontsize=9,
                    color="#888888",
                    style="italic",
                )

    # Label axes/title and attach a colorbar legend for the continuous color scale
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(param_strategies, fontsize=9)
    ax.set_yticklabels(data_objectives, fontsize=9)
    ax.set_xlabel("Parameter strategy", fontsize=10, fontweight="bold")
    ax.set_ylabel("Data objective", fontsize=10, fontweight="bold")
    ax.set_title(title, fontsize=11, fontweight="bold", pad=8)
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.04, label=label)


# Render the perplexity heatmap (left) and the parameter-cost heatmap (right)
draw_grid(
    axes[0],
    ppl_grid,
    "{:.1f}",
    "Held-Out Perplexity\n(lower = better)",
    "YlOrRd_r",
    "Perplexity",
)
draw_grid(
    axes[1],
    param_grid,
    "{:.2f}%",
    "Trainable Parameters %\n(lower = cheaper)",
    "Blues_r",
    "Trainable %",
)

plt.tight_layout()
plt.show()

#  Textual summary
print(f"\n{'=' * 80}")
print("Combination Grid — Key Takeaways:")
print(f"{'=' * 80}")

# Keep only trained cells, then sort them to find the best (lowest-perplexity) combination
trained_cells = [
    (r, c, ppl_grid[r, c]) for (r, c) in checkpoint_map if trained_mask[r, c]
]
trained_cells.sort(key=lambda x: x[2])
best_r, best_c, best_ppl = trained_cells[0]
print(
    f"  Best held-out perplexity: [{data_objectives[best_r].replace(chr(10),' ')}] × "
    f"[{param_strategies[best_c].replace(chr(10),' ')}] → {best_ppl:.1f}"
)
print()
print(
    "  Observation 1: Within continued pretraining (row 0), perplexity rises as params"
)
print(
    "    shrink — Full FT > Partial > LoRA — showing the quality/cost trade-off directly."
)
print()
print(
    "  Observation 2: Instruction-tuned and DPO models (rows 1-2) show HIGHER perplexity"
)
print(
    "    on raw prose — expected: they optimised for instruction-following, not next-token"
)
print("    prediction on plain paragraphs.")
print()
print("  Observation 3: The 4 grey cells (Instruction+FullFT, Instruction+Partial,")
print(
    "    DPO+FullFT, DPO+Partial) represent real engineering options — they would give"
)
print(
    "    instruction-following ability with more trainable capacity, at higher compute cost."
)
print(f"{'=' * 80}")

## Ablation Study: What Happens If You Skip a Stage?

**Riverside's question for this section:** the launch date got moved up. If we have to cut a corner
to ship on time, which corner is safe to cut, and which one breaks the assistant?

The three-stage pipeline (continued pretraining → instruction tuning → DPO) is sequential for a
reason. Let's explore **what breaks** if you skip stages or do them in the wrong order.

### Experiment 1: Skip Continued Pretraining (Base → Instruction Tuning Directly)

**Setup:** Train instruction tuning on a base model that has never seen the domain corpus.

**Expected result:**

- Model learns to follow the instruction format
- Model doesn't know domain vocabulary, characters, or settings
- Completions are generic and off-topic

**Example:**

Prompt: `"Continue: Aria Voss checked the Meridian's Promise and\n\n"`

Without domain pretraining:

> "she was surprised to find that the system was working perfectly. The crew had been working hard..."
> (Generic, no reference to story-specific elements)

With domain pretraining first:

> "found the quantum fold drive's containment field fluctuating at 3.2 terahertz. The Keeper's
> maintenance logs showed seventeen anomalies..." (Uses story-specific terminology)

**Verdict:** You **can** skip continued pretraining if your domain vocabulary overlaps heavily with
general English (e.g., customer support chatbot). You **cannot** skip it for specialized domains
(sci-fi, medical, legal) -- and Riverside's whole catalog is specialized domains. Skipping this stage
means shipping an assistant that can't answer the one question the company actually needs answered:
"who is Aria Voss?"

---

### Experiment 2: Skip Instruction Tuning (Continued Pretraining → DPO Directly)

**Setup:** Run DPO on a model that only knows domain text but hasn't been instruction-tuned.

**Expected result:**

- Model doesn't know how to "follow" a prompt/completion format
- DPO preferences are learned over random continuations, not helpful responses
- Model rambles without stopping

**Example:**

Prompt: `"Explain the five tides in the Tidebound Accord."`

Without instruction tuning:

> "Explain the five tides in the Tidebound Accord. The scholars of the Deepwater Academy have debated
> this question for centuries. Some argue there are six tides, others claim the void tide is merely
> theoretical. In the year 3847, the Council of Tidebound..." (rambles forever)

With instruction tuning first:

> "The five tides are water, wind, stone, flame, and void. Each corresponds to a fundamental force..."
> (Direct, stops after answering)

**Verdict:** You **must** do instruction tuning before DPO. DPO assumes the model already knows
instruction-following behavior. For Riverside's knowledge-base use case, this is the stage that turns
"a model that continues text" into "a model that answers an employee's question and then stops."

---

### Experiment 3: Wrong Order (DPO → Instruction Tuning)

**Setup:** Run DPO first (on a base model), then instruction tune afterward.

**Expected result:**

- DPO preferences are "erased" by subsequent instruction tuning
- Wastes compute (DPO training was for nothing)
- Final model behaves like it only had instruction tuning, no preference signal

**Why:** Instruction tuning updates the same parameters that DPO adjusted, and with a larger learning
rate / more data, it "overwrites" the preference alignment.

**Verdict:** Always do DPO **last**. The order matters because each stage builds on the previous one --
if Riverside's team ever re-trains after adding a new novel, this is the order to re-run the stages in,
every time.

---

### Experiment 4: Only Full Fine-Tuning, No Parameter Efficiency

**Setup:** Use full fine-tuning for all three stages on a 70B model.

**Expected result:**

- Maximum quality (model has full freedom to adapt)
- Requires 840 GB of GPU memory (unfeasible without multi-node setup)
- Takes 10x longer to train
- Higher risk of catastrophic forgetting

**Alternative:** Use LoRA throughout → 8 GB memory, 2% trainable params, ~90% of the quality.

**Verdict:** Full fine-tuning is only viable for small models (<7B) or when you have massive compute
budgets. LoRA is the production-standard approach for 13B+ models -- and the realistic choice if
Riverside ever wants to swap `gpt2-medium` for something bigger and better without buying a GPU
cluster.

---

### Experiment 5: Skip Fine-Tuning Entirely (Prompt Engineering Alone)

**Setup:** Someone in the room always asks this: why not skip training altogether and just write a
really good prompt? Ask the untouched base model a direct question, zero-shot, no fine-tuning of any
kind.

**Expected result:**

- The base model has no way to know about Riverside's private, unpublished catalog -- it was never
  shown a single page of it
- No amount of clever prompt phrasing can recover facts the model was never trained on
- The instruction-tuned checkpoint (continued pretraining + instruction tuning), by contrast, both
  knows the fact **and** answers in a direct, instruction-following format

**Example:**

Prompt: `"Who is Aria Voss?"`

Base model, zero-shot:

> A confident-sounding but generic or fabricated answer -- "made up" is the technical term for this
> failure mode, and it's exactly what you'd expect from Gap 1 (Domain Knowledge Gap) at the very top
> of this notebook: the base model has never seen this name, so it either invents a plausible-sounding
> answer or says it doesn't know.

Instruction-tuned model (same question, using the training format):

> A direct, on-topic answer describing Aria Voss's actual role in _The Weight of Distant Light_.

**Verdict:** Prompt engineering is genuinely powerful for reshaping behavior the model _already has_ --
tone, format, reasoning style. It cannot inject facts about content the base model has literally never
seen. This is the one gap from the very first "Three-Gap Problem" section that no amount of clever
prompting closes: only continued pretraining teaches the model that Aria Voss exists in the first
place.


### Putting Experiment 5 to the Test

Let's actually run the zero-shot prompt from Experiment 5 above instead of just reading the
hypothetical example, comparing the untouched base model against the instruction-tuned checkpoint we
trained earlier in this notebook.


In [ ]:
# Experiment 5 in practice: prompt engineering alone (base model) vs. fine-tuning
zero_shot_prompt = "Who is Aria Voss?"

print("=== Base model, zero-shot (no fine-tuning) ===")
print(generate(base_model, zero_shot_prompt), "\n")

print("=== Instruction-tuned model (same question, using the training format) ===")
print(generate(instruct_lora_model, INSTRUCTION_PREFIX + zero_shot_prompt + "\n\n"))

## What This Fine-Tuning Arc Covered (and What It Didn't)

Before the final decision below, here's a quick recap of everything Parts 1-3 actually demonstrated
on Riverside's corpus, and what was deliberately left out.

### Implemented and Demonstrated

**Data-based progression (Part 1, the journey):**

1. **Continued pretraining** - Absorb domain vocabulary, facts, and style
2. **Instruction tuning (SFT)** - Teach the model to follow instructions, not just continue text
3. **Preference alignment (DPO)** - Align outputs with human preferences beyond "technically correct"

**Parameter-based approaches (Part 2, the cost/quality trade-off):**

1. **Full fine-tuning** (100% params) - Maximum quality, maximum cost
2. **Partial freezing** (10-30% params) - Middle ground
3. **LoRA** (<1% params) - Minimum cost, swappable adapters
4. **QLoRA** (<1% params + 4-bit base) - explained in depth, contrasted with LoRA, but not trained
   here (GPU-specific -- see Part 2, Concept 7)

**Quantization for deployment (Part 2):** a real, executed post-training dynamic-quantization demo
(the only technique in that comparison table that's genuinely CPU-native), plus GPTQ/AWQ/static
PTQ/QAT compared honestly without being run.

**All five trainable combinations tested** on the same 7-novel corpus with side-by-side comparisons
(this notebook, Part 3).

### Mentioned but Not Fully Implemented

**Alternative preference alignment:**

- **PPO-based RLHF** with a separately-trained reward model and on-policy generation. The
  "DPO vs. PPO" comparison in Part 1's Concept 3 walks through how those mechanics differ from DPO's,
  in plain English and in a side-by-side table, but doesn't implement them in code -- unlike DPO,
  which Part 1 actually trains.

**Alternative parameter-efficient methods:**

- **Adapter layers** (bottleneck modules between transformer blocks)
- **Prefix/prompt tuning** (learnable virtual tokens prepended to input)
- **BitFit** (bias-only tuning)
- **IA3** (learned rescaling vectors)

(QLoRA moved out of this "not implemented" list in this revision -- Part 2's Concept 7 now covers it
in real depth, including the actual `BitsAndBytesConfig`/`prepare_model_for_kbit_training` code you'd
run on a GPU. It's still not _trained_ in this arc, for the reason given there.)

**Why these weren't included:**

- DPO is simpler and more practical than full PPO-based RLHF for Riverside's static-pairs situation --
  the comparison in Part 1 shows why: PPO needs a reward model and on-policy generation neither of
  which pays for itself when the preference data is already collected as pairs
- LoRA (and increasingly QLoRA, for larger models) has become the dominant PEFT approach in production
  (2024-2026)
- Other PEFT methods offer different trade-offs but similar principles

---

## Further Reading & Scaling Up

**To scale this fine-tuning arc:**

- **Larger corpus:** Set `max_chapters=None` to use all 197 chapters (~619K words)
- **More novels:** Add `.txt` files to `content/` and update the `NOVELS` dict
- **Bigger models:** Replace `gpt2-medium` with `gpt2-large`, `gpt2-xl`, etc. (will likely need a GPU)
- **Real datasets:**
  - Non-instructional: [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories),
    [openwebtext](https://huggingface.co/datasets/Skylion007/openwebtext)
  - Instructional: [alpaca-cleaned](https://huggingface.co/datasets/yahma/alpaca-cleaned),
    [OpenOrca](https://huggingface.co/datasets/Open-Orca/OpenOrca)
  - Preferences: [Anthropic/hh-rlhf](https://huggingface.co/datasets/Anthropic/hh-rlhf),
    [ultrafeedback-binarized](https://huggingface.co/datasets/argilla/ultrafeedback-binarized-preferences-cleaned)

**Key papers:**

- LoRA: [Hu et al. 2021](https://arxiv.org/abs/2106.09685)- RLHF: [Ouyang et al. 2022 (InstructGPT)](https://arxiv.org/abs/2203.02155)

- DPO: [Rafailov et al. 2023](https://arxiv.org/abs/2305.18290)- Instruction tuning: [Wei et al. 2021 (FLAN)](https://arxiv.org/abs/2109.01652)


## The Decision: What Do We Actually Hand to Riverside House?

We started with a brief: an in-house editing assistant and knowledge base, trained on proprietary
manuscripts that can never leave the building, on a laptop CPU. Here's what the evidence above
actually supports -- not a taxonomy recap, a recommendation.

### The scorecard

| Checkpoint                 | Held-out perplexity                               | Follows instructions?                    | Matches editor preference?                       | Cost to (re)train     |
| --------------------------- | -------------------------------------------------- | ------------------------------------------- | -------------------------------------------------- | --------------------- |
| Baseline (no fine-tuning)  | Worst                                             | No                                       | No                                               | Free                  |
| Full fine-tuning           | **Best**                                          | No (never instruction-tuned in this run) | No                                               | Highest (100% params) |
| Partial freezing           | 2nd best                                          | No (never instruction-tuned in this run) | No                                               | Medium (~21% params)  |
| LoRA continued pretraining | 3rd best                                          | No (never instruction-tuned in this run) | No                                               | Lowest (<1% params)   |
| Instruction-tuned (LoRA)   | Worse on raw prose (expected -- see caveat above) | **Yes**                                  | Not yet                                          | Lowest (<1% params)   |
| Preference-aligned (DPO)   | Worse on raw prose (expected)                     | Yes                                      | Attempted, **did not clearly converge this run** | Lowest (<1% params)   |

### What we'd actually ship

Riverside asked for two things, and this notebook's checkpoints map to them cleanly because DPO,
instruction tuning, and the parameter axis are **independent choices**:

1. **Knowledge base / house-style continuation** → deploy the **full-fine-tuning or partial-freezing
   checkpoint** (whichever the laptop's training-time budget allows day to day) -- these had the best
   held-out perplexity on Riverside's actual prose, which is exactly what "predict the next word in a
   house-style chapter" needs.
2. **Editing assistant that takes an instruction** → deploy the **instruction-tuned LoRA adapter**, not
   because it scored best on perplexity (it didn't, and we now know why), but because it's the only
   checkpoint that reliably stops after answering instead of rambling forever -- the specific gap
   Riverside's ghostwriters complained about.
3. **DPO is not ready to ship as-is.** The honest reflection earlier in the notebook stands: 30
   preference pairs and one pass wasn't enough signal in this run. Before Riverside turns this on for
   real editors, it needs real preference data -- probably from actual side-by-side ratings collected
   over a few weeks -- not a synthetic stand-in.
4. **Both deployed checkpoints should be LoRA adapters on one frozen base**, not full-fine-tuned or
   partially-frozen copies -- that's the one-frozen-base-model, swap-the-adapter architecture from the
   LoRA section, and it's what keeps Riverside from storing two 355M-parameter models when one plus
   two small adapters does the job.

### Key insights to keep

- **A held-out number beats a vibe check.** The qualitative comparisons earlier were useful for
  intuition, but they didn't reveal that instruction-tuning trades away raw next-token accuracy on
  plain prose -- the held-out perplexity table did.
- **The three data-based stages are cumulative, not competitive.** Skipping one doesn't just weaken
  the assistant, it removes a specific capability the notebook can point to (Ablation Study,
  Experiments 1-2).
- **The parameter axis is a budget decision, not a quality decision** -- LoRA gave up very little
  held-out perplexity for a >100x reduction in trainable parameters, which is the whole reason it won
  in production for models much bigger than `gpt2-medium`.
- **"It didn't work" is still a result.** The DPO run in this notebook is the most valuable evidence
  here precisely because it's honest: a real team would have hit the same wall with 30 preference
  pairs, and knowing that _before_ deployment is worth more than a cherry-picked success story.
- **Real mechanics beat illustrations.** Every number in the scorecard above -- the frozen blocks that
  measured exactly `0.00e+00` delta, the LoRA adapter's real forward-pass contribution, the held-out
  perplexity -- came from the actual trained models in this notebook's kernel, not a hand-picked
  example.

Riverside House doesn't get a perfect model from a laptop and a few minutes of training per stage --
but it gets an honest one, and now it knows exactly what it would take to make each piece
production-ready.

---

**This closes the three-part fine-tuning arc.** Riverside now has a decision on which checkpoints to
deploy -- but a trained model sitting on disk isn't a usable assistant yet. The next question is how
anyone actually _finds_ the right passage across 197 chapters to ground a question in, which is where
[`04-hybrid-search.ipynb`](04-hybrid-search.ipynb) picks up.


## From Notebook Decision to Production Control Plane

In production, the notebook's data-axis choice (continued pretraining → SFT → optional DPO) and parameter-axis choice (full, partial-freeze, or LoRA) become a versioned release decision rather than a one-time judgment. A candidate must pass workload-specific **evaluation gates** before promotion: held-out perplexity for house-style continuation, instruction-following for the editing assistant, preference win rate only when DPO is claimed, and safety/regression checks for both.[^1] The accepted base model, adapter/checkpoint, tokenizer, dataset fingerprint, code revision, seed, and measured metrics are then recorded together in an **artifact registry** so the exact release can be reconstructed.[^2]

The release gate must also include measured p95 latency and cost at the intended batch size and hardware; trainable-parameter percentage predicts training burden, but it does not replace serving benchmarks.[^3] Promotion should be gradual, with the prior immutable artifact retained as the rollback target. If any online quality, latency, cost, or safety threshold fails, routing returns to that known-good version without retraining.[^4]

The cells below define a small vendor-neutral manifest and gate runner. They default to `RUN_PRODUCTION_DECISION = False`, so reading or running the notebook does not hash checkpoints, read benchmark files, or write release artifacts unless explicitly enabled.

[^1]: NIST, [AI Risk Management Framework: Measure](https://airc.nist.gov/AI_RMF_Knowledge_Base/Playbook/Measure), recommends documented, repeatable evaluation against deployment-context criteria.
[^2]: MLflow's open-source [Model Registry concepts](https://mlflow.org/docs/latest/ml/model-registry/) illustrate versioned artifacts, lineage, aliases, and controlled promotion; the same metadata contract can be implemented with any registry.
[^3]: MLCommons, [MLPerf Inference](https://mlcommons.org/benchmarks/inference/), separates serving measurements by scenario because latency and throughput depend on the actual runtime configuration.
[^4]: Google SRE, [Canarying Releases](https://sre.google/workbook/canarying-releases/), describes comparing a candidate with a known-good release and halting or reverting when the canary degrades.

In [ ]:
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import random
from typing import Any, Mapping, Optional


RUN_PRODUCTION_DECISION = False


@dataclass(frozen=True)
class EvaluationPolicy:
    """Workload-specific promotion thresholds; replace defaults with service SLOs."""

    max_perplexity_regression_pct: Optional[float] = None
    min_instruction_pass_rate: Optional[float] = 0.95
    min_preference_win_rate: Optional[float] = None
    min_safety_pass_rate: float = 1.0
    max_p95_latency_ms: float = 1_500.0
    max_cost_per_1k_requests_usd: float = 1.00


@dataclass(frozen=True)
class ProductionDecisionConfig:
    workload: str = "editing-assistant"
    candidate_name: str = "Instruction-tuned (LoRA)"
    candidate_artifact: Path = Path("./checkpoints/instruction-lora")
    rollback_name: str = "previous-production"
    rollback_artifact: Path = Path("./artifacts/production/current")
    benchmark_metrics: Path = Path("./artifacts/production-benchmarks.json")
    registry_dir: Path = Path("./artifacts/finetuning-decisions")
    seed: int = 42
    policy: EvaluationPolicy = EvaluationPolicy()


def set_reproducible_seed(seed: int) -> None:
    """Seed the random sources used by this notebook's PyTorch workflow."""
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_benchmark_metrics(path: Path) -> dict[str, Any]:
    """Load offline quality, safety, latency, and cost measurements."""
    return json.loads(path.read_text(encoding="utf-8"))


def sha256_artifact(path: Path) -> str:
    """Create one deterministic digest for a checkpoint file or directory."""
    digest = hashlib.sha256()
    files = [path] if path.is_file() else sorted(item for item in path.rglob("*") if item.is_file())
    for file_path in files:
        relative_path = file_path.name if path.is_file() else file_path.relative_to(path).as_posix()
        digest.update(relative_path.encode("utf-8"))
        with file_path.open("rb") as artifact_file:
            for chunk in iter(lambda: artifact_file.read(1024 * 1024), b""):
                digest.update(chunk)
    return digest.hexdigest()


def evaluate_release(
    candidate: Mapping[str, float],
    baseline: Mapping[str, float],
    policy: EvaluationPolicy,
) -> dict[str, bool]:
    """Apply only the gates configured for this workload."""
    gates = {
        "instruction": (
            policy.min_instruction_pass_rate is None
            or candidate["instruction_pass_rate"] >= policy.min_instruction_pass_rate
        ),
        "preference": (
            policy.min_preference_win_rate is None
            or candidate["preference_win_rate"] >= policy.min_preference_win_rate
        ),
        "safety": candidate["safety_pass_rate"] >= policy.min_safety_pass_rate,
        "latency": candidate["p95_latency_ms"] <= policy.max_p95_latency_ms,
        "cost": candidate["cost_per_1k_requests_usd"] <= policy.max_cost_per_1k_requests_usd,
    }
    if policy.max_perplexity_regression_pct is not None:
        allowed = baseline["heldout_perplexity"] * (
            1.0 + policy.max_perplexity_regression_pct / 100.0
        )
        gates["perplexity"] = candidate["heldout_perplexity"] <= allowed
    return gates


def build_decision_manifest(
    config: ProductionDecisionConfig,
    benchmark: Mapping[str, Any],
    gates: Mapping[str, bool],
    artifact_digest: str,
) -> dict[str, Any]:
    """Capture the evidence and lineage needed to reproduce or roll back a release."""
    promoted = all(gates.values())
    return {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "workload": config.workload,
        "decision": "promote" if promoted else "rollback",
        "selected_name": config.candidate_name if promoted else config.rollback_name,
        "selected_artifact": str(
            config.candidate_artifact if promoted else config.rollback_artifact
        ),
        "candidate": {
            "name": config.candidate_name,
            "artifact": str(config.candidate_artifact),
            "sha256": artifact_digest,
        },
        "rollback": {
            "name": config.rollback_name,
            "artifact": str(config.rollback_artifact),
        },
        "reproducibility": {
            "seed": config.seed,
            "dataset_fingerprint": benchmark["dataset_fingerprint"],
            "code_revision": benchmark["code_revision"],
            "base_model": MODEL_NAME,
        },
        "policy": asdict(config.policy),
        "metrics": benchmark["candidate"],
        "baseline_metrics": benchmark["baseline"],
        "gates": dict(gates),
    }


def write_decision_manifest(manifest: Mapping[str, Any], registry_dir: Path) -> Path:
    """Write an immutable, content-addressed decision record."""
    registry_dir.mkdir(parents=True, exist_ok=True)
    canonical = json.dumps(manifest, sort_keys=True, separators=(",", ":"))
    decision_id = hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:12]
    output_path = registry_dir / f"decision-{decision_id}.json"
    output_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
    return output_path

In [ ]:
production_config = ProductionDecisionConfig()

if RUN_PRODUCTION_DECISION:
    set_reproducible_seed(production_config.seed)

    benchmark = load_benchmark_metrics(production_config.benchmark_metrics)
    candidate_metrics = dict(benchmark["candidate"])

    # Reuse this notebook's held-out result when the full comparison has already run.
    notebook_results = globals().get("holdout_results", {})
    if production_config.candidate_name in notebook_results:
        candidate_metrics["heldout_perplexity"] = notebook_results[
            production_config.candidate_name
        ]["perplexity"]

    benchmark = {**benchmark, "candidate": candidate_metrics}
    gates = evaluate_release(candidate_metrics, benchmark["baseline"], production_config.policy)

    if not production_config.candidate_artifact.exists():
        raise FileNotFoundError(
            f"Candidate artifact not found: {production_config.candidate_artifact}"
        )

    artifact_digest = sha256_artifact(production_config.candidate_artifact)
    manifest = build_decision_manifest(
        production_config,
        benchmark,
        gates,
        artifact_digest,
    )
    manifest_path = write_decision_manifest(manifest, production_config.registry_dir)

    for gate_name, passed in gates.items():
        print(f"{gate_name:>12}: {'PASS' if passed else 'FAIL'}")
    print(f"Decision: {manifest['decision'].upper()} -> {manifest['selected_name']}")
    print(f"Manifest: {manifest_path}")
else:
    print(
        "Production decision workflow is disabled. Set RUN_PRODUCTION_DECISION = True "
        "only after supplying measured benchmark metrics and an explicit rollback artifact."
    )